### Finviz Data Cleaning and Transformation Pipeline

This notebook processes the raw data downloaded from Finviz, cleans it, and transforms it into an analysis-ready format.

**Workflow:**

1.  **Load Data:** Reads a raw Finviz `.parquet` file for a specific date.
2.  **Feature Engineering:** Creates new composite columns (`Info`, `MktCap AUM`).
3.  **Data Type Conversion:**
    *   Converts currency strings (e.g., `1.5B`, `250K`) into numeric values in millions.
    *   Converts percentage strings (e.g., `12.5%`) into numeric values.
    *   Converts other object columns to their proper numeric types.
4.  **Final Processing:** Sorts the data by market capitalization, sets the `Ticker` as the index, and adds a `Rank` column.
5.  **Save & Verify:** Saves the cleaned DataFrame to a new `.parquet` file and verifies the saved file.

### Setup and Configuration

**This is the only cell you need to modify.** It contains all imports, paths, and lists of columns for processing.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- Project Path Setup ---
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Import config and custom utils now that path is set
from config import DATE_STR, DOWNLOAD_DIR, DEST_DIR
import utils

# --- File Path Configuration ---
# Build paths using pathlib for cross-platform compatibility
SOURCE_PATH = Path(DOWNLOAD_DIR) / f'df_finviz_{DATE_STR}_stocks_etfs.parquet'
DEST_PATH = Path(DEST_DIR) / f'{DATE_STR}_df_finviz_stocks_etfs.parquet'

# --- Column Processing Configuration ---
# Define which columns need specific cleaning operations.

# Columns to combine into the 'Info' column
INFO_COLS = ["Sector", "Industry", "Single Category", "Asset Type"]

# Columns with abbreviated currency values (B, M, K) to be converted to millions
CURRENCY_COLS = [
    'Market Cap', 'AUM', 'Sales', 'Income', 'Outstanding', 'Float', 
    'Short Interest', 'Avg Volume', 'Flows 1M', 'Flows 3M', 'Flows YTD',
    'MktCap AUM' # This is the new column we create
]

# Other columns that are numeric but stored as strings (objects)
# Note: Percentage columns are detected automatically in Step 3.
OTHER_NUMERIC_COLS = [
    "No.", "P/E", "Fwd P/E", "PEG", "P/S", "P/B", "P/C", "P/FCF",
    "Book/sh", "Cash/sh", "Dividend TTM", "EPS", "EPS next Q", "Short Ratio",
    "Curr R", "Quick R", "LTDebt/Eq", "Debt/Eq", "Beta", "ATR", "RSI",
    "Employees", "Recom", "Rel Volume", "Volume", "Target Price",
    "Prev Close", "Open", "High", "Low", "Price", "Holdings"
]

# --- Notebook Setup ---
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 2500)
%load_ext autoreload
%autoreload 2

# --- Verification ---
print(f"Source file: {SOURCE_PATH}")
print(f"Destination file: {DEST_PATH}")
print(f"Processing for date: {DATE_STR}")

Source file: C:\Users\ping\Downloads\df_finviz_2026-07-17_stocks_etfs.parquet
Destination file: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_finviz_stocks_etfs.parquet
Processing for date: 2026-07-17


### Step 1: Load Raw Data

Load the source Parquet file into a pandas DataFrame.

In [2]:
print(f"--- Step 1: Loading data from {SOURCE_PATH.name} ---")

try:
    df = pd.read_parquet(SOURCE_PATH, engine="pyarrow")
    print("Data loaded successfully.")
    df.info()
    display(df.head(3))
except FileNotFoundError:
    print(f"ERROR: Source file not found at {SOURCE_PATH}")
    df = None  # Ensure df is None if loading fails
except Exception as e:
    print(f"An error occurred during file loading: {e}")
    df = None

--- Step 1: Loading data from df_finviz_2026-07-17_stocks_etfs.parquet ---
Data loaded successfully.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Columns: 111 entries, No. to Tags
dtypes: float64(32), int64(2), object(77)
memory usage: 1.3+ MB


,No.,Ticker,Company,Index,Sector,Industry,Country,Exchange,Market Cap,P/E,Forward P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend,Dividend TTM,Dividend Ex Date,Payout Ratio,EPS,EPS next Q,EPS This Y,EPS Next Y,EPS Past 5Y,EPS Next 5Y,Sales Past 5Y,Sales Q/Q,EPS Q/Q,EPS YoY TTM,Sales YoY TTM,Sales,Income,EPS Surprise,Revenue Surprise,Outstanding,Float,Float %,Insider Own,Insider Trans,Inst Own,Inst Trans,Short Float,Short Ratio,Short Interest,ROA,ROE,ROIC,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M,Oper M,Profit M,Perf Week,Perf Month,Perf Quart,Perf Half,Perf Year,Perf YTD,Beta,ATR,Volatility W,Volatility M,SMA20,SMA50,SMA200,50D High,50D Low,52W High,52W Low,52W Range,All-Time High,All-Time Low,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open,Gap,Recom,Avg Volume,Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change,Single Category,Asset Type,Expense,Holdings,AUM,Flows 1M,Flows% 1M,Flows 3M,Flows% 3M,Flows YTD,Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags
0,461,LLVS,Las Vegas Sands Corp,S&P 500,Consumer Cyclical,Resorts & Casinos,USA,NYSE,30.06B,16.79,12.66,1.69,2.19,25.10,9.03,13.61,1.81,5.03,2.48%,1.10,5/5/2026,42.59%,2.70,0.76,9.52%,8.73%,-,7.51%,34.66%,25.26%,71.16%,50.79%,22.67%,13.74B,1.84B,20.31%,6.64%,663.00M,274.72M,41.44%,58.54%,-0.02%,42.23%,-0.86%,6.57%,3.90,18.05M,8.68%,94.53%,12.20%,0.92,0.91,11.60,13.13,38.37%,24.64%,13.41%,-2.85%,-7.32%,-20.01%,-24.27%,-7.30%,-30.31%,0.85,1.21,2.32%,2.47%,-2.42%,-7.64%,-19.10%,-16.95%,2.58%,-35.62%,2.58%,44.22 - 70.45,-67.54%,3399.34%,37.02,Jul 22/a,12/15/2004,Yes,Yes,41500.0,-0.50%,-0.24%,1.57,4.63M,0.93,4296079,66.73,45.70,45.59,45.70,45.02,45.36,-0.74%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-
1,462,FFTS,Fortis Inc,-,Utilities,Utilities - Regulated Electric,Canada,NYSE,29.92B,23.94,21.09,5.13,3.39,1.83,116.33,NaN,32.03,0.51,3.17%,1.83,5/15/2026,73.03%,2.46,0.55,0.19%,7.15%,4.66%,4.11%,5.51%,5.95%,3.40%,3.49%,5.00%,8.84B,1.24B,-0.32%,-3.73%,509.10M,507.93M,99.77%,0.23%,0.05%,56.50%,-1.68%,2.07%,14.76,10.49M,2.44%,7.62%,3.28%,0.49,0.41,1.28,1.45,28.43%,28.43%,14.05%,2.40%,2.91%,3.45%,13.19%,25.06%,13.17%,0.41,0.84,1.34%,1.39%,2.38%,3.95%,8.14%,0.19%,8.45%,0.00%,25.98%,46.66 - 58.78,0.00%,451.28%,62.10,Jul 31/b,11/19/2003,Yes,Yes,9900.0,-0.02%,0.20%,3.00,711.24K,1.93,1371159,57.26,58.67,58.79,59.08,58.24,58.78,0.19%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-
2,463,EERIC,Telefonaktiebolaget L M Ericsson ADR,-,Technology,Communication Equipment,Sweden,NASD,29.90B,12.56,14.80,NaN,1.23,3.02,5.35,9.16,3.25,1.84,3.28%,0.31,4/2/2026,-,0.78,0.15,-40.63%,18.28%,8.75%,-8.29%,-0.90%,-3.10%,-7.93%,56.42%,2.72%,24.30B,2.61B,4.54%,-3.24%,3.05B,3.04B,99.99%,0.00%,-,10.63%,-1.51%,2.14%,5.54,65.23M,8.98%,26.30%,19.10%,1.12,0.89,0.27,0.38,48.13%,13.75%,10.75%,-13.48%,-16.43%,-19.24%,4.69%,33.06%,1.76%,0.96,0.42,3.29%,2.69%,-10.30%,-17.74%,-8.70%,-28.69%,-0.23%,-28.69%,37.25%,7.16 - 13.77,-92.54%,1251.57%,32.71,Jul 14/b,8/24/1981,Yes,No,88826.0,-0.05%,-0.66%,3.04,11.78M,1.49,17534015,10.32,9.89,9.82,9.95,9.78,9.82,-0.71%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-


In [8]:
sorted(df["Ticker"].dropna().astype(str).tolist())

['AA',
 'AAA',
 'AAAL',
 'AAAOI',
 'AAAON',
 'AAAPL',
 'AABBV',
 'AABEV',
 'AABNB',
 'AABT',
 'AABVX',
 'AACA',
 'AACGL',
 'AACI',
 'AACM',
 'AACN',
 'AACWI',
 'AACWX',
 'AADBE',
 'AADC',
 'AADI',
 'AADM',
 'AADP',
 'AADSK',
 'AAEE',
 'AAEG',
 'AAEIS',
 'AAEM',
 'AAEP',
 'AAER',
 'AAES',
 'AAFG',
 'AAFL',
 'AAFRM',
 'AAG',
 'AAGCO',
 'AAGG',
 'AAGI',
 'AAGNC',
 'AAGX',
 'AAHR',
 'AAIG',
 'AAIQ',
 'AAIRR',
 'AAIT',
 'AAIZ',
 'AAJG',
 'AAKAM',
 'AAKRE',
 'AALAB',
 'AALB',
 'AALC',
 'AALGM',
 'AALGN',
 'AALKS',
 'AALL',
 'AALLE',
 'AALLY',
 'AALNY',
 'AALSN',
 'AALV',
 'AAM',
 'AAMAT',
 'AAMCR',
 'AAMD',
 'AAME',
 'AAMG',
 'AAMGN',
 'AAMH',
 'AAMKR',
 'AAMLP',
 'AAMP',
 'AAMRZ',
 'AAMT',
 'AAMX',
 'AAMZN',
 'AANET',
 'AAON',
 'AAOS',
 'AAPA',
 'AAPD',
 'AAPG',
 'AAPGE',
 'AAPH',
 'AAPLD',
 'AAPO',
 'AAPP',
 'AAPTV',
 'AAR',
 'AARCC',
 'AARE',
 'AARES',
 'AARGX',
 'AARKK',
 'AARM',
 'AARMK',
 'AARW',
 'AARWR',
 'AARXS',
 'AAS',
 'AASML',
 'AASND',
 'AASR',
 'AASTS',
 'AASX',
 'AATI',
 'AAT

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Columns: 111 entries, No. to Tags
dtypes: float64(32), int64(2), object(77)
memory usage: 1.3+ MB


In [ ]:
# Sort df by 'Market Cap' from largest to smallest
if df is not None:
  sort_col_m = "Market Cap, M"
  if sort_col_m in df.columns:
    df.sort_values(by=sort_col_m, ascending=False, inplace=True, na_position="last")
  else:
    tmp_sort = "_market_cap_sort_m"
    if "convert_to_millions" in globals():
      df[tmp_sort] = df["Market Cap"].apply(convert_to_millions)
    else:
      def _to_millions(val):
        if pd.isna(val):
          return np.nan
        s = str(val).strip().upper()
        if not s:
          return np.nan
        mul = {"T": 1_000_000, "B": 1_000, "M": 1, "K": 0.001}
        suf = s[-1]
        if suf in mul:
          try:
            return float(s[:-1]) * mul[suf]
          except Exception:
            return np.nan
        return np.nan
      df[tmp_sort] = df["Market Cap"].apply(_to_millions)

    df.sort_values(by=tmp_sort, ascending=False, inplace=True, na_position="last")
    df.drop(columns=[tmp_sort], inplace=True)

  print("Sorted by Market Cap (largest -> smallest).")
  display(df[["Market Cap"]].head(10))

In [9]:
df.head(20)

,No.,Ticker,Company,Index,Sector,Industry,Country,Exchange,Market Cap,P/E,Forward P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend,Dividend TTM,Dividend Ex Date,Payout Ratio,EPS,EPS next Q,EPS This Y,EPS Next Y,EPS Past 5Y,EPS Next 5Y,Sales Past 5Y,Sales Q/Q,EPS Q/Q,EPS YoY TTM,Sales YoY TTM,Sales,Income,EPS Surprise,Revenue Surprise,Outstanding,Float,Float %,Insider Own,Insider Trans,Inst Own,Inst Trans,Short Float,Short Ratio,Short Interest,ROA,ROE,ROIC,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M,Oper M,Profit M,Perf Week,Perf Month,Perf Quart,Perf Half,Perf Year,Perf YTD,Beta,ATR,Volatility W,Volatility M,SMA20,SMA50,SMA200,50D High,50D Low,52W High,52W Low,52W Range,All-Time High,All-Time Low,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open,Gap,Recom,Avg Volume,Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change,Single Category,Asset Type,Expense,Holdings,AUM,Flows 1M,Flows% 1M,Flows 3M,Flows% 3M,Flows YTD,Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags
0,461,LLVS,Las Vegas Sands Corp,S&P 500,Consumer Cyclical,Resorts & Casinos,USA,NYSE,30.06B,16.79,12.66,1.69,2.19,25.10,9.03,13.61,1.81,5.03,2.48%,1.10,5/5/2026,42.59%,2.70,0.76,9.52%,8.73%,-,7.51%,34.66%,25.26%,71.16%,50.79%,22.67%,13.74B,1.84B,20.31%,6.64%,663.00M,274.72M,41.44%,58.54%,-0.02%,42.23%,-0.86%,6.57%,3.90,18.05M,8.68%,94.53%,12.20%,0.92,0.91,11.60,13.13,38.37%,24.64%,13.41%,-2.85%,-7.32%,-20.01%,-24.27%,-7.30%,-30.31%,0.85,1.21,2.32%,2.47%,-2.42%,-7.64%,-19.10%,-16.95%,2.58%,-35.62%,2.58%,44.22 - 70.45,-67.54%,3399.34%,37.02,Jul 22/a,12/15/2004,Yes,Yes,41500.0,-0.50%,-0.24%,1.57,4.63M,0.93,4296079,66.73,45.70,45.59,45.70,45.02,45.36,-0.74%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-
1,462,FFTS,Fortis Inc,-,Utilities,Utilities - Regulated Electric,Canada,NYSE,29.92B,23.94,21.09,5.13,3.39,1.83,116.33,NaN,32.03,0.51,3.17%,1.83,5/15/2026,73.03%,2.46,0.55,0.19%,7.15%,4.66%,4.11%,5.51%,5.95%,3.40%,3.49%,5.00%,8.84B,1.24B,-0.32%,-3.73%,509.10M,507.93M,99.77%,0.23%,0.05%,56.50%,-1.68%,2.07%,14.76,10.49M,2.44%,7.62%,3.28%,0.49,0.41,1.28,1.45,28.43%,28.43%,14.05%,2.40%,2.91%,3.45%,13.19%,25.06%,13.17%,0.41,0.84,1.34%,1.39%,2.38%,3.95%,8.14%,0.19%,8.45%,0.00%,25.98%,46.66 - 58.78,0.00%,451.28%,62.10,Jul 31/b,11/19/2003,Yes,Yes,9900.0,-0.02%,0.20%,3.00,711.24K,1.93,1371159,57.26,58.67,58.79,59.08,58.24,58.78,0.19%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-
2,463,EERIC,Telefonaktiebolaget L M Ericsson ADR,-,Technology,Communication Equipment,Sweden,NASD,29.90B,12.56,14.80,NaN,1.23,3.02,5.35,9.16,3.25,1.84,3.28%,0.31,4/2/2026,-,0.78,0.15,-40.63%,18.28%,8.75%,-8.29%,-0.90%,-3.10%,-7.93%,56.42%,2.72%,24.30B,2.61B,4.54%,-3.24%,3.05B,3.04B,99.99%,0.00%,-,10.63%,-1.51%,2.14%,5.54,65.23M,8.98%,26.30%,19.10%,1.12,0.89,0.27,0.38,48.13%,13.75%,10.75%,-13.48%,-16.43%,-19.24%,4.69%,33.06%,1.76%,0.96,0.42,3.29%,2.69%,-10.30%,-17.74%,-8.70%,-28.69%,-0.23%,-28.69%,37.25%,7.16 - 13.77,-92.54%,1251.57%,32.71,Jul 14/b,8/24/1981,Yes,No,88826.0,-0.05%,-0.66%,3.04,11.78M,1.49,17534015,10.32,9.89,9.82,9.95,9.78,9.82,-0.71%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-
3,464,EEIX,Edison International,S&P 500,Utilities,Utilities - Regulated Electric,USA,NYSE,29.87B,8.42,11.91,6.58,1.52,1.73,177.81,NaN,44.80,0.44,4.51%,3.46,7/7/2026,29.09%,9.22,1.20,-6.49%,6.42%,42.36%,1.81%,7.31%,7.66%,-63.09%,30.35%,13.14%,19.61B,3.55B,7.58%,-1.21%,384.79M,384.11M,99.82%,0.17%,-1.46%,95.60%,-0.95%,3.56%,5.35,13.66M,4.05%,21.80%,6.65%,0.74,0.68,2.21,2.47,32.09%,27.90%,18.12%,3.23%,7.73%,8.42%,25.98%,51.24%,29.34%,0.65,1.65,2.55%,2.08%,3.45%,7.13%,17.87%,-0.83%,19.39%,-0.83%,53.48%,50.58 - 78.28,-12.55%,1142.08%,65.20,Jul 30/a,5/27/1926,Yes,Yes,13725.0,-1.99%,1.49%,2.67,2.55M,0.97,2483774,76.50,78.05,79.21,79.88,77.19,77.63,-0.54%,-,-,-,NaN,-,NaN,-,NaN,-,-,-,-,-,-,-
4,465,BBIDU,Baidu Inc ADR,-,Communication Services,Internet Content & Information,China,NASD,29.84B,NaN,12.74,1.49,1.65,0.94,1.75,NaN,114.39,61.12,1.66%,NaN,-,0.00%,-0.02,1.61,-8.82%,17.05%,-29.62%,8.54%,2.97%,3.87%,-51.49%,-98.39%,-2.40%,

### Step 2: Feature Engineering - Create Composite Columns

Combine existing columns to create more meaningful features: `Info` and `MktCap AUM`.

In [ ]:
if df is not None:
    print("\n--- Step 2: Engineering new features ---")

    # 1. Create 'Info' column by concatenating category columns.
    for col in INFO_COLS:
        if col in df.columns:
            df[col] = df[col].replace("-", "")
    df["Info"] = df[INFO_COLS].apply(
        lambda row: ", ".join(filter(None, row.astype(str))), axis=1
    )
    print("Created 'Info' column.")

    # 2. Create 'MktCap AUM' by concatenating 'Market Cap' and 'AUM'.
    # This combines stock and ETF liquidity metrics into a single string column for now.
    # It will be converted to numeric in the next step.
    df["MktCap AUM"] = df["Market Cap"].replace("-", "") + df["AUM"].replace("-", "")
    print("Created 'MktCap AUM' column.")

    # Display the new columns for verification
    display(df[["Ticker", "Info", "MktCap AUM"]].head(3))

### Step 3: Data Type Conversion

This multi-part step cleans and converts all string-based numeric and percentage columns into proper numeric types.

#### Part A: Convert Abbreviated Currency Columns to Millions

In [ ]:
def convert_to_millions(value: str) -> float:
    """Converts a string with a T/B/M/K suffix to a numeric value in millions."""
    if pd.isna(value):
        return np.nan

    value_str = str(value).strip().upper()
    if not value_str:
        return np.nan

    multipliers = {"T": 1_000_000, "B": 1_000, "M": 1, "K": 0.001}
    suffix = value_str[-1]

    if suffix in multipliers:
        number_part = value_str[:-1]
        try:
            return float(number_part) * multipliers[suffix]
        except (ValueError, TypeError):
            return np.nan
    return np.nan


if df is not None:
    print("\n--- Step 3a: Converting currency columns to millions ---")
    new_names = {}
    for col in CURRENCY_COLS:
        if col in df.columns:
            df[col] = df[col].apply(convert_to_millions)
            new_names[col] = f"{col}, M"

    df.rename(columns=new_names, inplace=True)
    print(f"Converted and renamed {len(new_names)} columns.")
    display(df[[name for name in new_names.values() if name in df.columns]].head(3))

#### Part B: Convert Percentage Columns to Numeric

In [ ]:
if df is not None:
    print("\n--- Step 3b: Converting percentage columns ---")
    percent_cols = [
        col
        for col in df.columns
        if df[col].dtype == "object" and df[col].str.endswith("%", na=False).any()
    ]

    if not percent_cols:
        print("No new percentage columns found to modify.")
    else:
        print("Processing the following percentage columns:")
        for col in percent_cols:
            df[col] = pd.to_numeric(df[col].str.replace("%", ""), errors="coerce")
            new_name = f"{col} %" if "%" not in col else col
            df.rename(columns={col: new_name}, inplace=True)
            print(f"  - Converted '{col}' to numeric and renamed to '{new_name}'")

#### Part C: Convert Other String-Based Numeric Columns

In [ ]:
if df is not None:
    print("\n--- Step 3c: Converting other numeric string columns ---")
    converted_count = 0
    for col in OTHER_NUMERIC_COLS:
        if col in df.columns and df[col].dtype == "object":
            df[col] = pd.to_numeric(
                df[col].str.replace(",", "", regex=False), errors="coerce"
            )
            converted_count += 1

    print(f"Converted {converted_count} additional columns to numeric type.")
    print("\nData types after all conversions:")
    df.info()

### Step 4: Final Processing - Sort, Index, and Rank

Sort the DataFrame by the unified liquidity metric, set the `Ticker` as the index, and add a final `Rank`.

In [ ]:
if df is not None:
    print("\n--- Step 4: Finalizing DataFrame ---")

    # 1. Sort by the primary metric in descending order
    df.sort_values(
        by="MktCap AUM, M", ascending=False, inplace=True, na_position="last"
    )
    print("Sorted DataFrame by 'MktCap AUM, M'.")

    # 2. Add a 'Rank' column based on the new sort order
    df["Rank"] = range(1, len(df) + 1)
    print("Added 'Rank' column.")

    # 3. Set 'Ticker' as the index
    if "Ticker" in df.columns:
        df.set_index("Ticker", inplace=True)
        print("Set 'Ticker' as the index.")

    print("\nFinal DataFrame structure:")
    display(df[["Rank", "Info", "MktCap AUM, M"]].head())

### Step 5: Save and Verify Cleaned Data

Save the final, cleaned DataFrame to a new Parquet file and read it back to verify integrity.

In [ ]:
if df is not None:
    print("\n--- Step 5: Saving and verifying data ---")
    try:
        # Ensure destination directory exists
        DEST_PATH.parent.mkdir(parents=True, exist_ok=True)

        # Save the file
        df.to_parquet(DEST_PATH, engine="pyarrow", compression="zstd")
        print(f"Successfully saved cleaned data to: {DEST_PATH}")

        # Verify by loading it back
        loaded_df = pd.read_parquet(DEST_PATH, engine="pyarrow")
        print("\nVerification successful. First 20 rows of the saved file:")
        display(loaded_df.head(20))

    except Exception as e:
        print(f"An error occurred during save or verification: {e}")